In [1]:
import pickle
import pandas as pd
import numpy as np
import os
from model_diffusion import eval_bagging
from results_anaylisis import show_table, prcess_and_save_xlsx

## features

In [2]:
0.3*15000, 0.2*15000

(4500.0, 3000.0)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_gnn_occsvm_pred'
out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/svm_feature_fused_5'
out_path_pred = out_path+'_pred'

os.makedirs(out_path, exist_ok=True)
os.makedirs(out_path_pred, exist_ok=True)

def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]

rate = 0.05
for filename in os.listdir(root):
    if '.pkl' in filename:
        with open(os.path.join(root, filename), "rb") as f:
            predictions = pickle.load(f)

        disease = filename.split('.')[0][:-5]
        prediction_collection = dict()

        result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

        threshold = int(rate*len(predictions['true_label']))
        weights = rank_weights(threshold)

        sort_dict = {
            k: np.argsort(predictions[k])[::-1]
            for k in predictions.keys()
            if k not in ['true_label','test_genes','train_pos_genes']
        }

        fuse_features = ['ppi_emb_df','ppi_emb_dw', 'ppi_emb_n2v', 'literature_emb','seq_emb_esm', 'seq_emb_port']
        feature_comb = 'feature_fused'
        fused_rank = np.zeros(len(predictions['true_label']), dtype=float)
        feature_scores = []
        for key in fuse_features:
            feature_scores.append(predictions[key])
            top_ranks_feat = sort_dict[key][:threshold]              # feature-specific
            # Map sample index -> weight for this feature
            # (vectorized)
            rank_positions = {idx: pos for pos, idx in enumerate(top_ranks_feat)}
            for sample_index in rank_positions:                       # only top 20% get nonzero
                fused_rank[sample_index] += weights[rank_positions[sample_index]]

        # fused_rank now reflects how consistently/highly a sample ranks across the selected features
        ranked_predict_index, results = eval_bagging(np.array(fused_rank), predictions['true_label'])
        prediction_collection[feature_comb] = np.array(fused_rank)
        result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'-0-0-0', *results]
        avg_lf = np.mean(np.array(feature_scores), axis=0)
        ranked_predict_index, results = eval_bagging(avg_lf, predictions['true_label'])
        prediction_collection[feature_comb+'_avg'] = avg_lf
        result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'_avg'+'-0-0-0', *results]
        

        with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
            pickle.dump(prediction_collection, f)

        result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

In [20]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/svm_feature_fused_5'
out_path = root+'.xlsx'
temp_df = pd.read_csv(os.path.join(root,os.listdir(root)[2]))

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
prcess_and_save_xlsx(fused_2019,all_avg_df,out_path)

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,feature_fused,0.092,0.338,0.491,0.580,0.734,0.293,0.143,0.304,0.381,0.498,0.0,0.0,0.0
random_negative,feature_fused_avg,0.081,0.269,0.530,0.769,0.819,0.181,0.130,0.286,0.385,0.577,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [18]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/svm_feature_fused_10'
out_path = root+'.xlsx'
temp_df = pd.read_csv(os.path.join(root,os.listdir(root)[2]))

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
prcess_and_save_xlsx(fused_2019,all_avg_df,out_path)

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,feature_fused,0.066,0.280,0.511,0.761,0.786,0.238,0.125,0.295,0.386,0.558,0.0,0.0,0.0
random_negative,feature_fused_avg,0.081,0.269,0.530,0.769,0.819,0.181,0.130,0.286,0.385,0.577,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [13]:
#0.2
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/svm_feature_fused'
out_path = root+'.xlsx'
temp_df = pd.read_csv(os.path.join(root,os.listdir(root)[2]))

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
prcess_and_save_xlsx(fused_2019,all_avg_df,out_path)

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,feature_fused,0.066,0.241,0.555,0.778,0.807,0.208,0.102,0.269,0.378,0.573,0.0,0.0,0.0
random_negative,feature_fused_avg,0.081,0.269,0.530,0.769,0.819,0.181,0.130,0.286,0.385,0.577,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
features_best = [0.308,	0.575,	0.821, 0.142,	0.309,	0.402]


## models

In [52]:
selection = {'df_gnn_occsvm_pred':['ppi_emb_n2v','GCNs','Graph_sage'],
            '2019_nn_all_pred':'ppi_emb_n2v',
            '2019_rf_renamed_pred':'ppi_emb_n2v',
            '2019_mf_bag_2_pred':'uniport_ppi_2019'}

out_path_pred = '/itf-fi-ml/shared/users/ziyuzh/svm/results/models_pred'
os.makedirs(out_path_pred, exist_ok=True)

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results'
init_path = os.path.join(root,list(selection.keys())[0])
for filename in os.listdir(init_path):
    models_pred = dict()
    with open(os.path.join(root, init_path, filename), "rb") as f:
        data_svm = pickle.load(f)
    models_pred['true_label'] = data_svm['true_label']
    models_pred['train_pos_genes'] = data_svm['train_pos_genes']
    models_pred['svm'] = data_svm['ppi_emb_n2v']
    models_pred['GCNs'] = data_svm['GCNs']
    models_pred['Graph_sage'] = data_svm['Graph_sage']

    for folder in list(selection.keys())[1:]:
        with open(os.path.join(root, folder, filename), "rb") as f:
            predictions = pickle.load(f)
        if not np.array_equal(predictions['test_genes'], data_svm['test_genes']):

            pred_genes = np.array(predictions['test_genes'])
            svm_genes = np.array(data_svm['test_genes'])
            gene_to_index = {gene: idx for idx, gene in enumerate(pred_genes)}
            reorder_idx = [gene_to_index[g] for g in svm_genes]

            predictions['true_label'] = np.array(predictions['true_label'])[reorder_idx]
            predictions['test_genes'] = pred_genes[reorder_idx]
            predictions['uniport_ppi_2019'] = np.array(predictions['uniport_ppi_2019'])[reorder_idx]
    
        models_pred[folder] = predictions[selection[folder]]
    
    with open(os.path.join(out_path_pred,filename), 'wb') as f:
        pickle.dump(models_pred, f)

In [18]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/models_pred'
def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]


feature_comb = 'model_fused'

for rate in [0.05,0.1,0.2]:
    out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/'+feature_comb+'_'+str(int(rate*100))
    out_path_pred = out_path+'_pred'

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(out_path_pred, exist_ok=True)

    for filename in os.listdir(root):
        if '.pkl' in filename:
            with open(os.path.join(root, filename), "rb") as f:
                predictions = pickle.load(f)

            disease = filename.split('.')[0][:-5]
            prediction_collection = dict()

            result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

            threshold = int(rate*len(predictions['true_label']))
            weights = rank_weights(threshold)

            sort_dict = {
                k: np.argsort(predictions[k])[::-1]
                for k in predictions.keys()
                if k not in ['true_label','test_genes','train_pos_genes']
            }

            fuse_features = ['svm', 'GCNs', 'Graph_sage', '2019_nn_all_pred', '2019_rf_renamed_pred', '2019_mf_bag_2_pred']
            fused_rank = np.zeros(len(predictions['true_label']), dtype=float)
            feature_scores = []
            for key in fuse_features:
                feature_scores.append(predictions[key])
                top_ranks_feat = sort_dict[key][:threshold]              # feature-specific
                # Map sample index -> weight for this feature
                # (vectorized)
                rank_positions = {idx: pos for pos, idx in enumerate(top_ranks_feat)}
                for sample_index in rank_positions:                       # only top 20% get nonzero
                    fused_rank[sample_index] += weights[rank_positions[sample_index]]

            # fused_rank now reflects how consistently/highly a sample ranks across the selected features
            ranked_predict_index, results = eval_bagging(np.array(fused_rank), predictions['true_label'])
            prediction_collection[feature_comb] = np.array(fused_rank)
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'-0-0-0', *results]
            avg_lf = np.mean(np.array(feature_scores), axis=0)
            ranked_predict_index, results = eval_bagging(avg_lf, predictions['true_label'])
            prediction_collection[feature_comb+'_avg'] = avg_lf
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'_avg'+'-0-0-0', *results]
            

            with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
                pickle.dump(prediction_collection, f)

            result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)
    temp_df = pd.read_csv(os.path.join(out_path,os.listdir(out_path)[2]))

    fused_2019, all_avg_df = show_table(out_path,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
    prcess_and_save_xlsx(fused_2019,all_avg_df,out_path+'.xlsx')

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,model_fused,0.059,0.275,0.522,0.559,0.733,0.275,0.121,0.278,0.362,0.486,0.0,0.0,0.0
random_negative,model_fused_avg,0.061,0.268,0.544,0.784,0.811,0.189,0.138,0.291,0.388,0.578,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,model_fused,0.059,0.274,0.537,0.737,0.785,0.230,0.121,0.278,0.375,0.552,0.0,0.0,0.0
random_negative,model_fused_avg,0.061,0.268,0.544,0.784,0.811,0.189,0.138,0.291,0.388,0.578,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,model_fused,0.059,0.295,0.528,0.766,0.803,0.220,0.122,0.280,0.379,0.569,0.0,0.0,0.0
random_negative,model_fused_avg,0.061,0.268,0.544,0.784,0.811,0.189,0.138,0.291,0.388,0.578,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
0,298	0,547	0,806	0,141	0,295	0,393


## models + features

In [7]:
selection = ['df_gnn_occsvm_pred',
            '2019_nn_all_pred',
            '2019_rf_renamed_pred']

out_path_pred = '/itf-fi-ml/shared/users/ziyuzh/svm/results/models_features_pred'
os.makedirs(out_path_pred, exist_ok=True)

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results'
init_path = os.path.join(root,selection[0])
for filename in os.listdir(init_path):
    models_pred = dict()
    with open(os.path.join(root, init_path, filename), "rb") as f:
        data_svm = pickle.load(f)
    for f in ['true_label', 'test_genes', 'train_pos_genes','ppi_emb_n2v', 'ppi_emb_dw', 'ppi_emb_df', 'literature_emb', 'seq_emb_esm', 'seq_emb_port']:
        models_pred[f] = data_svm[f]

    for folder in selection[1:]:
        with open(os.path.join(root, folder, filename), "rb") as f:
            predictions = pickle.load(f)
        if not np.array_equal(predictions['test_genes'], data_svm['test_genes']):
            print('mismatcg error')
        for b_f in ['ppi_emb_n2v', 'ppi_emb_dw', 'ppi_emb_df', 'literature_emb', 'seq_emb_esm', 'seq_emb_port']:
            models_pred[folder+'_'+b_f] = predictions[b_f]
    with open(os.path.join(out_path_pred,filename), 'wb') as f:
        pickle.dump(models_pred, f)

In [19]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/models_features_pred'
def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]

feature_comb = 'models_features_fused'

for rate in [0.05,0.1,0.2]:
    out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/'+feature_comb+'_'+str(int(rate*100))
    out_path_pred = out_path+'_pred'

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(out_path_pred, exist_ok=True)

    for filename in os.listdir(root):
        if '.pkl' in filename:
            with open(os.path.join(root, filename), "rb") as f:
                predictions = pickle.load(f)
            fuse_features = list(set(list(predictions.keys()))-set(['true_label', 'test_genes', 'train_pos_genes']))
            disease = filename.split('.')[0][:-5]
            prediction_collection = dict()

            result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

            threshold = int(rate*len(predictions['true_label']))
            weights = rank_weights(threshold)

            sort_dict = {
                k: np.argsort(predictions[k])[::-1]
                for k in predictions.keys()
                if k not in ['true_label','test_genes','train_pos_genes']
            }

            fused_rank = np.zeros(len(predictions['true_label']), dtype=float)
            feature_scores = []
            for key in fuse_features:
                feature_scores.append(predictions[key])
                top_ranks_feat = sort_dict[key][:threshold]              # feature-specific
                # Map sample index -> weight for this feature
                # (vectorized)
                rank_positions = {idx: pos for pos, idx in enumerate(top_ranks_feat)}
                for sample_index in rank_positions:                       # only top 20% get nonzero
                    fused_rank[sample_index] += weights[rank_positions[sample_index]]

            # fused_rank now reflects how consistently/highly a sample ranks across the selected features
            ranked_predict_index, results = eval_bagging(np.array(fused_rank), predictions['true_label'])
            prediction_collection[feature_comb] = np.array(fused_rank)
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'-0-0-0', *results]
            avg_lf = np.mean(np.array(feature_scores), axis=0)
            ranked_predict_index, results = eval_bagging(avg_lf, predictions['true_label'])
            prediction_collection[feature_comb+'_avg'] = avg_lf
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'_avg'+'-0-0-0', *results]
            
            with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
                pickle.dump(prediction_collection, f)

            result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

    temp_df = pd.read_csv(os.path.join(out_path,os.listdir(out_path)[2]))

    fused_2019, all_avg_df = show_table(out_path,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
    prcess_and_save_xlsx(fused_2019,all_avg_df,out_path+'.xlsx')

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,models_features_fused,0.059,0.294,0.484,0.691,0.757,0.300,0.138,0.292,0.374,0.529,0.0,0.0,0.0
random_negative,models_features_fused_avg,0.082,0.276,0.527,0.790,0.825,0.175,0.138,0.290,0.386,0.579,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,models_features_fused,0.071,0.265,0.495,0.755,0.796,0.218,0.124,0.283,0.375,0.557,0.0,0.0,0.0
random_negative,models_features_fused_avg,0.082,0.276,0.527,0.790,0.825,0.175,0.138,0.290,0.386,0.579,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,models_features_fused,0.064,0.258,0.499,0.762,0.807,0.196,0.110,0.264,0.363,0.558,0.0,0.0,0.0
random_negative,models_features_fused_avg,0.082,0.276,0.527,0.790,0.825,0.175,0.138,0.290,0.386,0.579,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
features_best = [0.308,	0.575,	0.821, 0.142,	0.309,	0.402]

## fusion + models

In [16]:
selection = ['df_gnn_occsvm_pred',
            '2019_nn_all_pred']

out_path_pred = '/itf-fi-ml/shared/users/ziyuzh/svm/results/fusion_models_pred'
os.makedirs(out_path_pred, exist_ok=True)

selected_features = ['early_fused', 'late_fused', 'mid_fused','mid_linear_fused','mid_geo_fused']

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results'
init_path = os.path.join(root,selection[0])
for filename in os.listdir(init_path):
    models_pred = dict()
    with open(os.path.join(root, init_path, filename), "rb") as f:
        data_svm = pickle.load(f)
    a = ['true_label', 'test_genes', 'train_pos_genes']
    a.extend(selected_features)

    for f in a:
        if f in data_svm.keys():
            models_pred[f] = data_svm[f]

    for folder in selection[1:]:
        with open(os.path.join(root, folder, filename), "rb") as f:
            predictions = pickle.load(f)
        if not np.array_equal(predictions['test_genes'], data_svm['test_genes']):
            print('mismatcg error')
        for b_f in selected_features:
            if b_f in predictions.keys():
                models_pred[folder+'_'+b_f] = predictions[b_f]
    with open(os.path.join(out_path_pred,filename), 'wb') as f:
        pickle.dump(models_pred, f)

In [21]:
selection = ['df_gnn_occsvm_pred',
            '2019_nn_all_pred']

out_path_pred = '/itf-fi-ml/shared/users/ziyuzh/svm/results/fusion_models_ppi_pred'
os.makedirs(out_path_pred, exist_ok=True)

selected_features = ['early_fused_ppi', 'late_fused_ppi','mid_fused_ppi', 'mid_geo_fused_ppi', 'mid_linear_fused_ppi']

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results'
init_path = os.path.join(root,selection[0])
for filename in os.listdir(init_path):
    models_pred = dict()
    with open(os.path.join(root, init_path, filename), "rb") as f:
        data_svm = pickle.load(f)
    a = ['true_label', 'test_genes', 'train_pos_genes']
    a.extend(selected_features)

    for f in a:
        if f in data_svm.keys():
            models_pred[f] = data_svm[f]

    for folder in selection[1:]:
        with open(os.path.join(root, folder, filename), "rb") as f:
            predictions = pickle.load(f)
        if not np.array_equal(predictions['test_genes'], data_svm['test_genes']):
            print('mismatcg error')
        for b_f in selected_features:
            if b_f in predictions.keys():
                models_pred[folder+'_'+b_f] = predictions[b_f]
    with open(os.path.join(out_path_pred,filename), 'wb') as f:
        pickle.dump(models_pred, f)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/fusion_models_pred'
def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]

feature_comb = 'fusion_models_fused'

for rate in [0.05,0.1,0.2]:
    out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/'+feature_comb+'_'+str(int(rate*100))
    out_path_pred = out_path+'_pred'

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(out_path_pred, exist_ok=True)

    for filename in os.listdir(root):
        if '.pkl' in filename:
            with open(os.path.join(root, filename), "rb") as f:
                predictions = pickle.load(f)
            fuse_features = list(set(list(predictions.keys()))-set(['true_label', 'test_genes', 'train_pos_genes']))
            disease = filename.split('.')[0][:-5]
            prediction_collection = dict()

            result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

            threshold = int(rate*len(predictions['true_label']))
            weights = rank_weights(threshold)

            sort_dict = {
                k: np.argsort(predictions[k])[::-1]
                for k in predictions.keys()
                if k not in ['true_label','test_genes','train_pos_genes']
            }

            fused_rank = np.zeros(len(predictions['true_label']), dtype=float)
            feature_scores = []
            for key in fuse_features:
                feature_scores.append(predictions[key])
                top_ranks_feat = sort_dict[key][:threshold]              # feature-specific
                # Map sample index -> weight for this feature
                # (vectorized)
                rank_positions = {idx: pos for pos, idx in enumerate(top_ranks_feat)}
                for sample_index in rank_positions:                       # only top 20% get nonzero
                    fused_rank[sample_index] += weights[rank_positions[sample_index]]

            # fused_rank now reflects how consistently/highly a sample ranks across the selected features
            ranked_predict_index, results = eval_bagging(np.array(fused_rank), predictions['true_label'])
            prediction_collection[feature_comb] = np.array(fused_rank)
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'-0-0-0', *results]
            avg_lf = np.mean(np.array(feature_scores), axis=0)
            ranked_predict_index, results = eval_bagging(avg_lf, predictions['true_label'])
            prediction_collection[feature_comb+'_avg'] = avg_lf
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'_avg'+'-0-0-0', *results]
            
            with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
                pickle.dump(prediction_collection, f)

            result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

    temp_df = pd.read_csv(os.path.join(out_path,os.listdir(out_path)[2]))

    fused_2019, all_avg_df = show_table(out_path,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
    prcess_and_save_xlsx(fused_2019,all_avg_df,out_path+'.xlsx')

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_fused,0.061,0.279,0.537,0.591,0.74,0.306,0.133,0.288,0.374,0.505,0.0,0.0,0.0
random_negative,fusion_models_fused_avg,0.079,0.261,0.557,0.777,0.82,0.180,0.136,0.281,0.378,0.574,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_fused,0.054,0.294,0.529,0.711,0.776,0.257,0.127,0.285,0.382,0.553,0.0,0.0,0.0
random_negative,fusion_models_fused_avg,0.079,0.261,0.557,0.777,0.820,0.180,0.136,0.281,0.378,0.574,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_fused,0.054,0.267,0.560,0.767,0.796,0.233,0.124,0.277,0.380,0.570,0.0,0.0,0.0
random_negative,fusion_models_fused_avg,0.079,0.261,0.557,0.777,0.820,0.180,0.136,0.281,0.378,0.574,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
fusion_best = [0.3,	0.554,	0.821,	0.157,	0.314, 0.402]

top_recall_10%  merge top20%

In [22]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/fusion_models_ppi_pred'
def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]

feature_comb = 'fusion_models_ppi_fused'

for rate in [0.05,0.1,0.2]:
    out_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/'+feature_comb+'_'+str(int(rate*100))
    out_path_pred = out_path+'_pred'

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(out_path_pred, exist_ok=True)

    for filename in os.listdir(root):
        if '.pkl' in filename:
            with open(os.path.join(root, filename), "rb") as f:
                predictions = pickle.load(f)
            fuse_features = list(set(list(predictions.keys()))-set(['true_label', 'test_genes', 'train_pos_genes']))
            disease = filename.split('.')[0][:-5]
            prediction_collection = dict()

            result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

            threshold = int(rate*len(predictions['true_label']))
            weights = rank_weights(threshold)

            sort_dict = {
                k: np.argsort(predictions[k])[::-1]
                for k in predictions.keys()
                if k not in ['true_label','test_genes','train_pos_genes']
            }

            fused_rank = np.zeros(len(predictions['true_label']), dtype=float)
            feature_scores = []
            for key in fuse_features:
                feature_scores.append(predictions[key])
                top_ranks_feat = sort_dict[key][:threshold]              # feature-specific
                # Map sample index -> weight for this feature
                # (vectorized)
                rank_positions = {idx: pos for pos, idx in enumerate(top_ranks_feat)}
                for sample_index in rank_positions:                       # only top 20% get nonzero
                    fused_rank[sample_index] += weights[rank_positions[sample_index]]

            # fused_rank now reflects how consistently/highly a sample ranks across the selected features
            ranked_predict_index, results = eval_bagging(np.array(fused_rank), predictions['true_label'])
            prediction_collection[feature_comb] = np.array(fused_rank)
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'-0-0-0', *results]
            avg_lf = np.mean(np.array(feature_scores), axis=0)
            ranked_predict_index, results = eval_bagging(avg_lf, predictions['true_label'])
            prediction_collection[feature_comb+'_avg'] = avg_lf
            result_df.loc[len(result_df.index)] = ["random_negative",'1',feature_comb+'_avg'+'-0-0-0', *results]
            
            with open(out_path_pred+f'/{disease}_pred.pkl', 'wb') as f:
                pickle.dump(prediction_collection, f)

            result_df.to_csv(os.path.join(out_path, f"{disease}.csv"),index = False)

    temp_df = pd.read_csv(os.path.join(out_path,os.listdir(out_path)[2]))

    fused_2019, all_avg_df = show_table(out_path,ban_list=[],fnum=len(temp_df),fused = True,input_weights = True)
    prcess_and_save_xlsx(fused_2019,all_avg_df,out_path+'.xlsx')

method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_ppi_fused,0.066,0.307,0.469,0.508,0.711,0.306,0.144,0.297,0.364,0.460,0.0,0.0,0.0
random_negative,fusion_models_ppi_fused_avg,0.060,0.299,0.567,0.772,0.821,0.179,0.145,0.308,0.404,0.589,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_ppi_fused,0.066,0.319,0.571,0.700,0.782,0.253,0.141,0.305,0.400,0.560,0.0,0.0,0.0
random_negative,fusion_models_ppi_fused_avg,0.060,0.299,0.567,0.772,0.821,0.179,0.145,0.308,0.404,0.589,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


method,para,top_recall_25,top_recall_300,top_recall_10%,top_recall_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights_1,weights_2,weights_3
random_negative,fusion_models_ppi_fused,0.064,0.293,0.547,0.755,0.791,0.245,0.137,0.300,0.399,0.576,0.0,0.0,0.0
random_negative,fusion_models_ppi_fused_avg,0.060,0.299,0.567,0.772,0.821,0.179,0.145,0.308,0.404,0.589,0.0,0.0,0.0


11


/itf-fi-ml/shared/users/ziyuzh/svm/src/results_anaylisis.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
fusion_ppi_best = [0.321,	0.592,	0.824,	0.15,	0.311, 0.41] 
## open fused results see if any metrics better

In [ ]:
df_gnn_occsvm_pred

svm_feature_fused
models_features_fused
fusion_models_fused
fusion_models_ppi_fused

In [3]:
features_best = [0.308,	0.575,	0.821, 0.142,	0.309,	0.402]
fusion_best = [0.3,	0.554,	0.821,	0.157,	0.314, 0.402]
fusion_ppi_best = [0.321,	0.592,	0.824,	0.15,	0.311, 0.41] 
metrics = ['top_recall_300','top_recall_10%', 'auroc', 'bedroc_1','bedroc_5', 'bedroc_10']


In [5]:
import pandas as pd
import os

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results'
dfs = []
for rate in ['5', '10', '20']:
    source = f"svm_feature_fused_{rate}"
    path = os.path.join(root, f"{source}.xlsx")

    df = pd.read_excel(path, sheet_name="all (macro avg)")
    df["source"] = source
    dfs.append(df)

stacked = pd.concat(dfs, ignore_index=True)
stacked = stacked.set_index("source")
stacked = stacked[metrics]

In [7]:
dfs

[Empty DataFrame
 Columns: [para, top_recall_25, top_recall_300, top_recall_10%, top_recall_30%, auroc, rank_ratio, bedroc_1, bedroc_5, bedroc_10, bedroc_30, weights_1, weights_2, weights_3, source]
 Index: [],
 Empty DataFrame
 Columns: [para, top_recall_25, top_recall_300, top_recall_10%, top_recall_30%, auroc, rank_ratio, bedroc_1, bedroc_5, bedroc_10, bedroc_30, weights_1, weights_2, weights_3, source]
 Index: [],
 Empty DataFrame
 Columns: [para, top_recall_25, top_recall_300, top_recall_10%, top_recall_30%, auroc, rank_ratio, bedroc_1, bedroc_5, bedroc_10, bedroc_30, weights_1, weights_2, weights_3, source]
 Index: []]